## Dataset Overview
We loaded 100 call transcripts and classified all 100 with a local Llama model. Every chart below uses those 100 rows. The model assigned call type, category, sentiment, topics, products, and escalation risk.

In [1]:
import pandas as pd
import ast
import matplotlib.pyplot as plt
from collections import Counter

df = pd.read_csv("../data/results.csv")

df["secondary_topics"] = df["secondary_topics"].apply(
    lambda x : ast.literal_eval(x) if isinstance(x, str) else []
)
df['products_mentioned'] = df['products_mentioned'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)
print(f"Total meetings: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
df.head()


Total meetings: 100
Columns: ['meeting_id', 'title', 'host', 'start_time', 'end_time', 'duration', 'attendees', 'date', 'duration_min', 'sentence_count', 'call_type', 'primary_topic', 'secondary_topics', 'category', 'sentiment_score', 'sentiment_label', 'escalation_risk', 'llm_summary', 'evidence', 'is_business_call', 'products_mentioned', 'confidence_score']


,meeting_id,title,host,start_time,end_time,duration,attendees,date,duration_min,sentence_count,...,secondary_topics,category,sentiment_score,sentiment_label,escalation_risk,llm_summary,evidence,is_business_call,products_mentioned,confidence_score
0,01KQ7F58A2E07B857C24624D,Competitive Landscape Review,jordan.whitfield@aegiscloud.com,2026-02-19T09:15:00.000Z,2026-02-19T09:35:12.000Z,20.2,"['jordan.whitfield@aegiscloud.com', 'tom.bradl...",2026-02-19,20.2,36,...,"[SentinelShield, VaultEdge, CyberNova, Fortigu...",Internal Planning,0.7,neutral,False,Tom and Jordan discuss the competitive landsca...,"Jordan: ""So from what I've been able to piece ...",True,"[Detect, Protect, Comply, Identity, SentinelSh...",0.9
1,01KQE110900B8A65D80AE694,Detect Outage - Root Cause Analysis,sofia.petrov@aegiscloud.com,2026-03-14T14:30:00.000Z,2026-03-14T14:45:54.000Z,15.9,"['sofia.petrov@aegiscloud.com', 'mike.romano@a...",2026-03-14,15.9,38,...,"[Detect outage, Circuit Breaker Pattern, Inade...",Incident Response,-0.8,negative,True,The team discussed a recent Detect outage that...,Sofia Petrov: 'Six hours — where customers had...,True,[Aegis Detect],0.9
2,01KQ5A966832A146DA4B7D41,Aegis / Summit Trust - Platform Concerns Discu...,maria.santos@aegiscloud.com,2026-03-26T16:00:00.000Z,2026-03-26T16:53:00.000Z,53.0,"['maria.santos@aegiscloud.com', 'a.monroe@summ...",2026-03-26,53.0,47,...,"[Comply v2, Detect outage]",Technical Support,0.6,negative,True,Alicia Monroe discusses issues with Aegis Iden...,Alicia mentions 'inconsistent enforcement of M...,True,"[Aegis Identity, Aegis Detect, Comply v2]",0.9
3,01KQ1A6B7E81B06F4A13B60D,Support Case #6977 - Brightpath Commerce Slow ...,david.kim@aegiscloud.com,2026-02-21T11:30:00.000Z,2026-02-21T11:45:48.000Z,15.8,"['david.kim@aegiscloud.com', 'k.wallace@bright...",2026-02-21,15.8,35,...,"[Aegis Protect, Incremental backup jobs, Backu...",Technical Support,0.2,negative,True,Customer experienced slow backup performance w...,"Kendra Wallace: Yeah, so — I mean, it's been t...",True,[Aegis Protect],1.0
4,01KQ77F2867FC54E13A8C495,Aegis / Brightpath Commerce - Detect Module De...,james.mitchell@aegiscloud.com,2026-02-11T09:15:00.000Z,2026-02-11T10:00:48.000Z,45.8,"['james.mitchell@aegiscloud.com', 'k.wallace@b...",2026-02-11,45.8,44,...,"[Aegis Detect deployment for Brightpath, Aegis...",Onboarding,0.8,positive,False,James Mitchell walked Kendra Wallace through t...,"James Mitchell: ""So the plan for today — I wan...",True,[Aegis Detect],0.9


In [2]:
df['call_type'].value_counts().plot(
    kind='bar',
    title='Meetings by Call Type'
)

plt.ylabel("Count")
plt.xlabel("Call Type")
plt.tight_layout()

plt.show()

/var/folders/7f/z48g1k9x2zqggf9q87j92rxw0000gn/T/ipykernel_68248/4192623312.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Internal calls are the largest group (42), then support (38), then external (20). Support and external together are 58 of 100, so customer-facing meetings are still a majority, but external account calls are the smallest slice.

**Why it matters:** Later charts have to keep call type in the cut. Averaging the corpus treats a short support ticket and a long internal planning meeting as the same kind of call.

These charts use the model label. The dashboard reviews 17 of them: one email domain is internal, a support-case title is support, and a customer on an account, onboarding, or feedback call is external. That moves the mix to support 39, external 31, and internal 30. Support stays the stressed queue in both cuts.

## Sentiment Analysis
Sentiment was extracted per transcript on a scale of -1.0 (very negative) 
to 1.0 (very positive). Support calls are expected to skew negative — 
customers call when something is wrong. The interesting finding is whether 
external or internal calls also show stress signals.

In [3]:
order = ['negative', 'neutral', 'positive']
df['sentiment_label'].value_counts()[order].plot(
    kind='bar',
    title='Overall Meeting Sentiment',
    figsize=(7, 4)
)
plt.ylabel("Number of Meetings")
plt.xlabel("Sentiment")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

/var/folders/7f/z48g1k9x2zqggf9q87j92rxw0000gn/T/ipykernel_68248/2070496904.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** 47 meetings are positive, 30 are negative, and 23 are neutral. Of the 30 negative meetings, 24 are support and 6 are internal. External calls contribute none.

**Why it matters:** The overall bar is mostly positive. That average hides a support queue that is mostly negative, which is what the next chart separates.

In [4]:
pd.crosstab(
    df['call_type'],
    df['sentiment_label']
).plot(
    kind='bar',
    stacked=True,
    figsize=(9, 5),
    title='Sentiment Distribution by Call Type'
)

plt.ylabel("Number of Meetings")
plt.xlabel("Call Type")
plt.xticks(rotation=0)
plt.legend(title='Sentiment', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

/var/folders/7f/z48g1k9x2zqggf9q87j92rxw0000gn/T/ipykernel_68248/2322222611.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Support is the stressed queue: 24 of 38 calls are negative (63%) and 6 are positive. External calls are the calm queue: 16 of 20 are positive, 4 are neutral, and none are negative. Internal calls are mostly positive (25 of 42), with 11 neutral and 6 negative.

**Why it matters:** Customer stress in this sample shows up in support, not in account-manager calls. The 6 negative internal meetings are the team responding to incidents, not a second customer queue. Reading external sentiment as the health of the account base would miss the support load.

In [5]:
pd.crosstab(
    df['call_type'],
    df['escalation_risk']
).plot(
    kind='bar',
    stacked=True,
    figsize=(9, 5),
    title='Escalation Risk by Call Type'
)

plt.ylabel("Number of Meetings")
plt.xlabel("Call Type")
plt.xticks(rotation=0)
plt.legend(title='Escalation Risk', labels=['No Risk', 'At Risk'], 
           bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

/var/folders/7f/z48g1k9x2zqggf9q87j92rxw0000gn/T/ipykernel_68248/2613033929.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Escalation risk is 31 of 38 support calls (82%), 9 of 42 internal calls (21%), and 0 of 20 external calls. No external meeting in this file is flagged. The internal flags are incident bridges and reviews, not renewal calls.

**Why it matters:** The escalation count belongs to support. Sending that queue to customer success, on the theory that external calls are the churn risk, does not match these labels.

In [6]:
pd.crosstab(
    df['sentiment_label'],
    df['escalation_risk']
).plot(
    kind='bar',
    stacked=True,
    figsize=(8, 5),
    title='Escalation Risk Across Sentiment Levels'
)

plt.ylabel("Number of Meetings")
plt.xlabel("Sentiment")
plt.tight_layout()
plt.show()

/var/folders/7f/z48g1k9x2zqggf9q87j92rxw0000gn/T/ipykernel_68248/1354312552.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** All 30 negative calls are flagged as escalation risk. Positive calls almost never are (2 of 47). The gap is the neutral band: 8 of 23 neutral calls are still flagged, 5 support and 3 internal. Those include open support cases and post-incident reviews where the tone stays polite.

**Why it matters:** An alert that fires only on `sentiment_label == negative` drops those 8 meetings. Risk has to be read next to sentiment, not inferred from it.

## Why LLM-Based Categorization
I chose LLM-based categorization over rule-based or clustering because:
- Dataset is small (100 transcripts) — clustering needs more data to work reliably
- Business topics require semantic understanding, not keyword matching  
- LLM handles nuance naturally — "billing dispute" vs "billing question" are different intents
- Categories can be refined by updating the prompt, not retraining a model

In [7]:
df['category'].value_counts().plot(
    kind='barh',
    figsize=(9, 5),
    title='Meetings by Category'
)

plt.xlabel("Number of Meetings")
plt.tight_layout()
plt.show()

/var/folders/7f/z48g1k9x2zqggf9q87j92rxw0000gn/T/ipykernel_68248/2847529063.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Technical Support (34) and Account Management (25) are the bulk of the corpus. Incident Response is smaller (11) and 9 of those 11 are escalated (82%). Technical Support holds 25 of the 40 escalations. Onboarding is uniformly positive: all 6 calls, mean sentiment +0.84. Internal Planning has no escalations (0 of 18) and a mean sentiment of +0.81.

**Why it matters:** Volume and risk rank categories differently. Account Management is the second-largest category and a low-risk one (4 of 25 escalated). Incident Response is easy to under-staff if categories are sorted by count alone.

In [9]:
import json
from pathlib import Path

RAW_DIR = Path("../data/raw")

CATEGORY_CUES = {
    "Product Feedback": ["feature request", "feedback", "would love", "wish"],
    "Billing": ["invoice", "billing", "seat", "charge", "overage"],
    "Onboarding": ["onboarding", "kickoff", "getting started", "setup"],
    "Incident Response": ["outage", "incident", "bridge", "war room"],
    "Internal Planning": ["roadmap", "deployment plan", "priorit", "planning"],
    "Account Management": ["renewal", "account review", "qbr"],
    "Technical Support": ["not working", "error", "slow", "failing", "alert", "bug", "issue"],
}

def load_turns(meeting_id):
    path = RAW_DIR / meeting_id / "transcript.json"
    return json.loads(path.read_text())["data"]

def cue_start(turns, cues):
    for i, turn in enumerate(turns):
        sentence = turn["sentence"].lower()
        if len(sentence.split()) >= 8 and any(cue in sentence for cue in cues):
            return i
    return 0

def format_turns(turns, start, max_turns=4, max_chars=240):
    lines = []
    for turn in turns[start:start + max_turns]:
        sentence = " ".join(turn["sentence"].split())
        if len(sentence) > max_chars:
            sentence = sentence[:max_chars].rsplit(" ", 1)[0] + "…"
        lines.append(f"  {turn['speaker_name']}: {sentence}")
    return "\n".join(lines)

sorted_categories = df["category"].value_counts().sort_values(ascending=True).index

print("=== ONE TRANSCRIPT EXCERPT PER CATEGORY ===")
print("Quoted from data/raw/<meeting_id>/transcript.json\n")
for category in sorted_categories:
    cues = CATEGORY_CUES.get(category, [])
    best = None
    for _, row in df[df["category"] == category].iterrows():
        turns = load_turns(row["meeting_id"])
        text = " ".join(turn["sentence"] for turn in turns).lower()
        score = sum(text.count(cue) for cue in cues)
        rank = (score, row["confidence_score"])
        if best is None or rank > best[0]:
            best = (rank, row, turns)
    _, sample, turns = best
    print(f"CATEGORY: {category}")
    print(f"Title:     {sample['title']}")
    print(f"Call type: {sample['call_type']}")
    print(f"Meeting:   {sample['meeting_id']}")
    print(format_turns(turns, cue_start(turns, cues)))
    print("-" * 60)

=== ONE TRANSCRIPT EXCERPT PER CATEGORY ===
Quoted from data/raw/<meeting_id>/transcript.json

CATEGORY: Product Feedback
Title:     Support Case #8811 - Blackridge Investments Custom Compliance Template
Call type: support
Meeting:   01KQ92EA85FE4C537400227D
  Priya Patel: Absolutely, no that's — unusual is fine, we love those. So I've got the case here, case number 8811, and it looks like you're coming in with a feature request around custom compliance templates in Aegis Comply. Is that the right starting…
  Warren Beck: That's exactly it. So, you know, we upgraded to Comply v2 — gosh, what was it, like mid-April? — and honestly the on-demand reporting has been phenomenal. The SOC 2 and the PCI DSS reports especially, our audit team is basically over the…
  Priya Patel: Oh that's great to hear, we launched v2 on April 7th so yeah you're right around that mark. And yeah the on-demand piece was a big lift for our engineering team so I'm glad it's landing well.
  Warren Beck: Yeah it rea

## Bonus Insights

Four cuts sit on top of category and sentiment. The charts measure the claim in the finding under them.

### 1. Detect outage, by week
Meetings that mention both Detect and an outage cluster in support and internal calls during the weeks of 9 Mar and 16 Mar. External calls do not move with that spike.
**Useful for:** Engineering leads and support leaders sizing one incident.

### 2. Products under stress
Stress is the share of a product's meetings that are negative or escalated, with aliases such as "Detect" and "Aegis Detect" grouped. The monthly chart is a separate question: which labels are being discussed, and when.
**Useful for:** Product managers separating a launch people talk about from a product that shows up in bad calls.

### 3. Duration versus escalation
Mean duration split by call type and escalation risk. The long meetings are internal and external, and they are mostly not escalated.
**Useful for:** Support leaders deciding whether call length is a frustration signal.

### 4. Escalation inside neutral calls
8 of 23 neutral calls are still flagged (the sentiment-by-escalation chart above). A polite label is not a safe call.
**Useful for:** Support leads who triage on tone.

In [9]:
df["date"] = pd.to_datetime(df["date"])
blob = (
    df["title"].fillna("")
    + " "
    + df["primary_topic"].fillna("")
    + " "
    + df["secondary_topics"].apply(lambda topics: " ".join(topics))
).str.lower()
detect = df[blob.str.contains("detect") & blob.str.contains("outage")].copy()
detect["week"] = detect["date"].dt.to_period("W").astype(str)

weekly = (
    detect.groupby(["week", "call_type"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["support", "internal", "external"], fill_value=0)
)
all_weeks = df["date"].dt.to_period("W").astype(str).drop_duplicates().sort_values()
weekly = weekly.reindex(all_weeks, fill_value=0)

weekly.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 5),
    title="Detect + Outage Meetings by Week and Call Type",
    color=["#E84C4C", "#E8A44C", "#4C9BE8"],
)
plt.ylabel("Number of Meetings")
plt.xlabel("Week")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Call Type")
plt.tight_layout()
plt.show()

print(f"Meetings mentioning Detect and an outage: {len(detect)}")
print(weekly.loc[weekly.sum(axis=1) > 0].to_string())

Meetings mentioning Detect and an outage: 44
call_type              support  internal  external
date                                              
2026-02-16/2026-02-22        0         0         1
2026-03-09/2026-03-15        7         4         0
2026-03-16/2026-03-22        6         2         0
2026-03-23/2026-03-29        2         3         1
2026-03-30/2026-04-05        0         3         0
2026-04-06/2026-04-12        1         3         0
2026-04-13/2026-04-19        1         3         0
2026-04-20/2026-04-26        2         3         0
2026-04-27/2026-05-03        0         1         1


/var/folders/7f/z48g1k9x2zqggf9q87j92rxw0000gn/T/ipykernel_68248/3221837866.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** The Detect outage is a two-week spike in support and internal calls. The week of 9 Mar has 7 support meetings and 4 internal meetings. The week of 16 Mar adds 6 support and 2 internal. External calls do not rise with it: only 3 external meetings mention Detect and an outage at all, on 17 Feb, 24 Mar, and 28 Apr.

**Why it matters:** The blast radius in this file is the support queue plus the internal bridges next to it. Calling it a renewal-conversation event overstates the external calls. The same two weeks are the sentiment trough further down: 9–22 Mar averages −0.35 sentiment and a 91% escalation rate, against +0.59 and 26% in the rest of the dataset.

In [10]:
from collections import Counter
import ast

def parse_products(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return []

def product_family(name):
    label = str(name).lower()
    if "detect" in label:
        return "Detect"
    if "comply" in label:
        return "Comply"
    if "identity" in label:
        return "Identity"
    if "protect" in label:
        return "Protect"
    return None

all_products = []
for products in df["products_mentioned"].dropna():
    all_products.extend(parse_products(products))
product_counts = Counter(all_products)

stressed = (df["sentiment_label"] == "negative") | (df["escalation_risk"] == True)
rows = []
for family in ["Detect", "Comply", "Identity", "Protect"]:
    mentioned = df["products_mentioned"].apply(
        lambda products, family=family: any(product_family(p) == family for p in products)
    )
    total = int(mentioned.sum())
    stress_n = int((mentioned & stressed).sum())
    rows.append({
        "product": family,
        "meetings": total,
        "stressed_meetings": stress_n,
        "stressed_share": stress_n / total,
    })

share_df = pd.DataFrame(rows).sort_values("stressed_share")
plt.figure(figsize=(9, 4))
plt.barh(share_df["product"], share_df["stressed_share"] * 100)
plt.xlabel("Share of that product's meetings that are negative or escalated (%)")
plt.title("Product Stress Rate (label aliases grouped)")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

print(share_df.sort_values("stressed_share", ascending=False).to_string(index=False))

/var/folders/7f/z48g1k9x2zqggf9q87j92rxw0000gn/T/ipykernel_68248/634623198.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 product  meetings  stressed_meetings  stressed_share
  Detect        67                 31        0.462687
Identity        30                 10        0.333333
  Comply        73                 21        0.287671
 Protect        30                  7        0.233333


In [11]:
exploded = (df
    .assign(
        month=pd.to_datetime(df['date']).dt.to_period('M'),
        products_parsed=df['products_mentioned'].apply(parse_products)
    )
    .explode('products_parsed')
    .rename(columns={'products_parsed': 'product'})
)

top_products = [p for p, _ in product_counts.most_common(5)]
monthly = (exploded[exploded['product'].isin(top_products)]
    .groupby(['month', 'product'])
    .size()
    .unstack(fill_value=0)
    .sort_index())

monthly.plot(kind='bar', figsize=(12, 5),
             title='Product Mentions by Month (Top 5 Products)')
plt.xlabel('Month')
plt.ylabel('Number of Calls')
plt.legend(title='Product', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

/var/folders/7f/z48g1k9x2zqggf9q87j92rxw0000gn/T/ipykernel_68248/147833048.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Friction and volume are different rankings. With aliases grouped, Detect is mentioned in 67 meetings and 31 of those are negative or escalated (46%). Comply is mentioned in more meetings (73) but only 21 are stressed (29%). Identity is 10 of 30 (33%). Protect is 7 of 30 (23%). Detect is the product most often attached to a bad call. Comply is the product mentioned in the most meetings.

The monthly bars use raw labels, so "Aegis Detect" and "Detect" stay split. On the Aegis Detect label the path is 9 in February, 27 in March, and 18 in April. That March bar is the outage. Comply v2 is 7, then 14, then 26, up every month and still rising after the outage. Identity stays at 10, 10, 9. Protect dips to 6 in March and is 11 in April.

**Why it matters:** A total-mention ranking would send attention to whichever product was merely discussed. The stress rate points support and incident work at Detect. The monthly trend points launch coverage at Comply v2.

In [12]:
means = (
    df.groupby(["call_type", "escalation_risk"])["duration_min"]
    .mean()
    .unstack()
    .reindex(["support", "external", "internal"])
    .rename(columns={False: "No escalation", True: "Escalation risk"})
)
means.plot(
    kind="bar",
    figsize=(8, 5),
    title="Average Duration by Call Type and Escalation Risk",
)
plt.ylabel("Average Duration (Minutes)")
plt.xlabel("Call Type")
plt.xticks(rotation=0)
plt.legend(title=None)
plt.tight_layout()
plt.show()

print(means.round(1).to_string())
print("\nDuration by call type:")
print(df.groupby("call_type")["duration_min"].describe().round(1))

escalation_risk  No escalation  Escalation risk
call_type                                      
support                   18.9             24.1
external                  35.1              NaN
internal                  35.5             30.8

Duration by call type:
           count  mean   std   min   25%   50%   75%   max
call_type                                                 
external    20.0  35.1  10.1  21.3  27.8  34.0  43.5  54.1
internal    42.0  34.5  10.7  15.9  26.0  34.8  42.9  54.3
support     38.0  23.1  11.5   9.3  14.0  20.8  29.4  53.0


/var/folders/7f/z48g1k9x2zqggf9q87j92rxw0000gn/T/ipykernel_68248/301615353.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Long meetings are internal and external, not support. Mean duration is 35.1 minutes for external (20 calls), 34.5 for internal (42), and 23.1 for support (38). The 75th percentile is 43.5, 42.9, and 29.4. Support's average is pulled down by short tickets such as Support Case #8811 (9.3 minutes), Trailhead Detect Alerts (10.0), and Brightpath backup performance (15.8).

Length does not track escalation across the file. Escalated meetings average 25.6 minutes and the others average 33.5 (correlation −0.32), because the long internal and external calls are usually not escalated. External has no escalated meetings at all. Inside support the gap is small: escalated support calls average 24.1 minutes, against 18.9 for the 7 that are not escalated. A 30-minute cutoff is a weak flag. Only 9 support calls run past 30 minutes, and 8 of those 9 are escalated, but 23 of the 31 escalated support calls are still under 30 minutes. The long support outliers are Summit Trust at 53 minutes, Northstar Pharma Detect impact at 48, and Quantum Edge renewal concerns at 45.

**Why it matters:** Staffing on the idea that angry support calls run long does not match this data. Most escalated support calls are short. The meetings that take a block of calendar are internal planning and external account calls.

In [13]:
all_topics = [topic for topics in df['secondary_topics'] for topic in topics]
top_topics = Counter(all_topics).most_common(10)

labels, values = zip(*top_topics)

plt.figure(figsize=(9, 5))
plt.barh(labels, values)
plt.title("Top 10 Themes Across All Calls")
plt.xlabel("Frequency")
plt.tight_layout()
plt.show()

/var/folders/7f/z48g1k9x2zqggf9q87j92rxw0000gn/T/ipykernel_68248/1461736855.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** On the raw secondary-topic labels, Comply v2 appears 50 times, HIPAA compliance 37, and Detect outage 27. The outage is split across extra labels the bar chart does not merge: Aegis Detect outage (5), Aegis Detect (6), and Detect (1). Added to the 27, Detect-related theme labels total 39, just above HIPAA. Aegis Protect is 11, with no incident attached to that label. SentinelShield is 7. It is a competitor name, and it shows up in internal competitive reviews plus a few renewal conversations, not as an Aegis product. PCI DSS and SOC 2 are 4 each, so compliance pressure is wider than HIPAA and Comply v2 alone.

**Why it matters:** Three subjects dominate the raw labels: a product launch (Comply v2), a regulation (HIPAA), and an incident (Detect outage). A Q3 plan that drops any one of those is dropping a large share of what these calls are about. The Detect total is undercounted if the split labels are left separate.

In [14]:

df['date'] = pd.to_datetime(df['date'])


sentiment_over_time = df.groupby('date')['sentiment_score'].mean()

plt.figure(figsize=(12, 5))
plt.plot(sentiment_over_time.index, sentiment_over_time.values, 
         marker='o', linewidth=2)
plt.title("Average Sentiment Over Time")
plt.ylabel("Sentiment Score (-1 to 1)")
plt.xlabel("Date")
plt.axhline(y=0, color='red', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

/var/folders/7f/z48g1k9x2zqggf9q87j92rxw0000gn/T/ipykernel_68248/1929932971.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Sentiment is steady and positive except for the Detect outage window. The week of 9 Mar is the trough: 13 meetings, mean sentiment −0.62. The week of 16 Mar is 0.04 across 9 meetings. From 9–22 Mar the mean is −0.35 and the escalation rate is 91% (22 meetings). Outside that window the mean is +0.59 and the escalation rate is 26%.

**Why it matters:** The time series is the same incident as the weekly Detect chart, seen as customer tone. One fortnight accounts for the negative swing in an otherwise positive corpus.

In [15]:
high_risk = df[df['escalation_risk'] == True]

high_risk['category'].value_counts().plot(
    kind='barh',
    figsize=(9, 5),
    title=f'High Risk Meetings by Category ({len(high_risk)} total)'
)

plt.xlabel("Number of Meetings")
plt.tight_layout()
plt.show()

/var/folders/7f/z48g1k9x2zqggf9q87j92rxw0000gn/T/ipykernel_68248/2692583301.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** 40 meetings are flagged. Technical Support is 25 of those 40. Incident Response is 9, and 8 of those 9 fall in 10–20 Mar around the Detect outage (the ninth is the 7 Apr reliability review). Account Management is 4: Quantum Edge and Nova Retail in March, Ironclad on 28 Mar, Brightpath on 22 Apr. Billing is 2, both April support cases.

Together, Technical Support and Incident Response are 34 of 40. The rates differ from the counts: Incident Response is escalated in 9 of 11 meetings (82%), Technical Support in 25 of 34 (74%).

**Why it matters:** Most escalated meetings are ordinary support cases, so the count points at support triage. Incident Response is a smaller set with a higher hit rate, and it is the same March outage as the sentiment dip. Dropping either view understates the risk.

In [16]:
low_confidence = df[df['confidence_score'] < 0.6]
print(f"Low confidence meetings: {len(low_confidence)}")
print(low_confidence[['title', 'call_type', 'confidence_score']])

Low confidence meetings: 0
Empty DataFrame
Columns: [title, call_type, confidence_score]
Index: []


**Finding:** No call scored under 0.6. Across the 100 results the minimum confidence is 0.8 and the mean is 0.93.

**Why it matters:** The validator accepts a confidence score from 0 to 1. The 0.6 line is a review check in this notebook, and it does not produce a manual-review queue. Every score in the file is at least 0.8. High confidence means the model committed. A spot-check of a few calls against `call_type` and `category` is still the right check before presenting the totals.